Processar o arquivo de base com as perguntas.

In [12]:
# Carregar as questões a serem usadas, cuja lógica está em outro arquivo.
# Realizar download do arquivo direto do GitHub.
!wget https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb -O questions.ipynb

# Executar o notebook de base na íntegra.
%run questions.ipynb

--2026-04-03 19:59:45--  https://raw.githubusercontent.com/eduoududu/juridico/refs/heads/main/questions.ipynb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35314 (34K) [text/plain]
Saving to: ‘questions.ipynb’

questions.ipynb     100%[===================>]  34.49K  --.-KB/s    in 0.002s  

2026-04-03 19:59:45 (14.9 MB/s) - ‘questions.ipynb’ saved [35314/35314]



Instalação de pacotes, seguido de importação de bibliotecas e funções necessárias.

In [13]:
# Instalar ou atualizar as bibliotecas necessárias.
!pip install -q -U openai

# Importar bibliotecas.
import pandas as pd
import os
from google.colab import userdata
from openai import OpenAI

Funcões dos clientes das IAs, modelos e realização de consultas.

In [31]:
# Iniciar o cliente da IA
def client_ai_instance(groq_api_key):
  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )
  return client_ai

def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def questions_submit(client_ai, model_id, df_questions):
    # Criar lista vazia para guardar as respostas.
    responses = []

    # Repetição para percorrer todo Dataframe.
    for index, row in df_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      questao = row['question']
      papel = row['system']
      categoria = row['category']
      contexto = row['statement']
      dificuldade = row['level']
      instrucao = row['turns']

      # Preencher prompt do usuário.
      prompt_usuario = markdown_prepare(papel, categoria, contexto, dificuldade, instrucao)

      # Realizar consulta a IA.
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          max_tokens=1024,
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

      # Armazenar o resultado em uma lista.
      responses.append({"question": questao, "response": response})
      print(f"Questão {questao} processada com sucesso.")

    # Fechar conexão com a IA (somente se você a usou ativamente)
    client_ai.close()

    # por motivo de performance as repostas foram adicionadas a uma lista.
    # No retorno a lista é convertida para um dataframe.
    return pd.DataFrame(responses)

Realizar consulta usando o modelo llama 3 de 70 bilhões de parâmetros.

In [32]:
# Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
# Tal chave foi previamente criada no Console do Groq, site: console.groq.com
groq_api_key = userdata.get('groq_api_key')

# Modelo llama 3 de 70 bilhões de parâmetros.
model_id = 'llama-3.3-70b-versatile'

# Dataframe com as respostas do primeiro modelo
llama_responses = questions_submit(client_ai_instance(groq_api_key), model_id, df_my_questions)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn7rg2zeee3t7f7drnrps2kb` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99013, Requested 2363. Please try again in 19m48.864s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Realizar consulta via modelo GPT-OSS de 20 bilhões de parâmetros.

In [23]:
# Modelo gpt-oss de 20 bilhões de parâmetros.
model_id = 'qwen/qwen3-32b'

# Dataframe com as respostas do primeiro modelo
gpt_oss_responses = questions_submit(client_ai_instance(groq_api_key), model_id, df_my_questions)

Questão 31 processada com sucesso.
Questão 32 processada com sucesso.
Questão 33 processada com sucesso.
Questão 34 processada com sucesso.
Questão 35 processada com sucesso.
Questão 36 processada com sucesso.
Questão 37 processada com sucesso.
Questão 38 processada com sucesso.
Questão 39 processada com sucesso.
Questão 40 processada com sucesso.


Modelo qwen 3 de 32 bilhões de parâmetros.

In [34]:
# Modelo llama 4 scout, mais leve, de 17 bilhões de parâmetros.
model_id = 'meta-llama/llama-4-scout-17b-16e-instruct'

# Dataframe com as respostas do primeiro modelo
qwen_responses = questions_submit(client_ai_instance(groq_api_key), model_id, df_my_questions)

,question,response
0,31,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
1,32,O prazo para oferta dos embargos à execução fi...
2,33,## Resposta\n\nO Imposto Sobre Serviços (ISS) ...
3,34,A exigência de depósito do montante integral c...
4,35,## Resposta\n\n### Item 1: Correção do Argumen...
5,36,EXCELENTÍSSIMO SENHOR DOUTOR JUIZ DE DIREITO D...
6,37,A responsabilização administrativa de pessoas ...
7,38,A questão apresentada envolve a interpretação ...
8,39,"Sim, é possível a utilização do sistema de reg..."
9,40,A aprovação de Iná no concurso público para a ...
